In [ ]:
"""
Created By    : Jared W. Marquis
Creation Date : 01 August 2022
Course        : ATSC 528 - Atmospheric Data Analysis
Assignment    : #03 - Statistical Objective Analysis

Purpose:
Script to take sparse upper air observations and analyze them on a
polar stereographic map projection and run the analysis through a shallow
water model.
[PUT MORE INFORMATION HERE - I.E., WHAT SPECIFIC THING IS BEING DONE]

"""
__author__    = "Jared W. Marquis"
__contact__   = "jared.marquis@und.edu"

In [ ]:
### Import Required Modules (shouldn't need to change) ###
import numpy as np                 #numpy for math
import matplotlib.pyplot as plt    #matplotlib for plotting
import cartopy.crs as ccrs         #cartopy for plotting on map
import cartopy.feature as cfeature #cartopy basic shapefiles
from shallow_water import shallow_water

### Needed Functions ###
- to_h_points(u, v):
    - this averages the u and v wind fields to the height grid for plotting
    - u is the staggered u-wind field
    - v is the staggered v-wind field
    - returns uh and vh which are the u- and v-wind components for the height grid
    - Note: this assumes a C-grid
###### ######
- geostrophic_c(h, dx, dy, f, sig_u, sig_v, g=9.80665):
    - calculates the geostrophic wind for the staggered grid
    - h is an array the height field in meters
    - dx is the value of x-step size on the image plane
    - dy is the value of y-step size on the image plane
    - f is the value of the coriolis force (single value)
    - sig_u is an array of sigma (image scale factor) for the u-staggered grid
    - sig_v is an array of sigma (image scale factor) for the v-staggered grid
    - g is the value for gravity (defaults to 9.80665)

In [ ]:
def to_h_points(u, v):
    """
    Average staggered grid u and v to the interior h points [1:-1, 1:-1].
    Assumes C-grid.

    inputs:
      u  = the staggered u wind field
      v  = the staggered v wind field

    returns:
      uh = the staggered u wind field averaged to the height grid
      vh = the staggered v wind field averaged to the height grid
    """
    
    uh = 0.5*(u[1:, 1:-1] + u[:-1, 1:-1])   # u south and north of each h point
    vh = 0.5*(v[1:-1, 1:] + v[1:-1, :-1])   # v east and west of each h point
    return uh, vh 

def geostrophic_c(h, dx, dy, f_u, f_v, sig_u, sig_v, g=9.80665):
    """
    Grid-relative geostrophic wind at the C-grid u and v points.
    dx, dy in meters on the image plane.

    inputs:
      h     = height field (in meters)
      dx    = x-interval in meters on the image plane (before map scaling)
      dy    = y-interval in meters on the image plane (before map scaling)
      f_u   = Corilois parameter at the u-staggered-grid (in /s)
      f_v   = Corilois parameter at the v-staggered-grid (in /s)
      sig_u = image scale factor at the u-staggered-grid 
      sig_v = image scale factor at the v-staggered-grid
      g     = gravity (defaults to 9.80665 m/s)


    returns:
      ug    = u geostrophic wind at the u-staggered grid
      vg    = v geostrophic wind at the v-staggered grid
    """
    
    dhdy_h = np.gradient(h, dy, axis=1)                  # at h points (x_n, y_n)
    dhdx_h = np.gradient(h, dx, axis=0)

    dhdy_u = 0.5*(dhdy_h[1:, :] + dhdy_h[:-1, :])        # to u points (x_n-1, y_n)
    dhdx_v = 0.5*(dhdx_h[:, 1:] + dhdx_h[:, :-1])        # to v points (x_n, y_n-1)

    ug = -(g*sig_u/f_u) * dhdy_u
    vg =  (g*sig_v/f_v) * dhdx_v
    return ug, vg

### Reuse Your Barnes Function from the Before ###


In [ ]:
#barnes function

In [ ]:
### Read in observations ###


In [ ]:
### Convert Winds from Direction & Speed (kts) to U and V m/s ###
### Hint: Check out the Polar Stereographic Document (Eqns A.8)


In [ ]:
### Set up analysis map with a 22x28 rectangular grid of points ###


### Set up staggered Arakawa C Grid ###
**Arakawa C-grid cell** (x increases south, y increases east)

$$
\begin{array}{ccccc l}
h_{i,j} & \text{———} & v_{i,j} & \text{———} & h_{i,j+1} & \quad x = x_i \\
\vert & & & & \vert & \\
u_{i,j} & & & & u_{i,j+1} & \quad x = x_i + \Delta x/2 \\
\vert & & & & \vert & \\
h_{i+1,j} & \text{———} & v_{i+1,j} & \text{———} & h_{i+1,j+1} & \quad x = x_{i+1}
\end{array}
$$


In [ ]:
#make the u- and v-wind grids


In [ ]:
### convert obs lat/long to x,y ###


### Use the code below to verify your grid looks correct ###
This assumes your map scale is given by "m" and the 2d grid variables are:
- Y & X for the height grid
- Yu & Xu for the u-wind grid
- Yv & Xv for the v-wind grid

In [ ]:
### Plot analysis grid, staggered grid, and observations ###
globe = ccrs.Globe(ellipse=None, semimajor_axis=rho/100, semiminor_axis=rho/100)
proj = ccrs.Stereographic(central_longitude=lambda_0,central_latitude=90,true_scale_latitude=phi_0,globe=globe)
fig = plt.figure(figsize=(8,8),dpi=150)
ax1 = fig.add_subplot(1,1,1,projection=proj)
ax1.add_feature(cfeature.STATES)
ax1.add_feature(cfeature.COASTLINE)
#plot height grid points
ax1.scatter(Y/m/100,-X/m/100,marker='*',s=20,color='k',transform=proj,label='analysis points')
#plot u grid points
ax1.scatter(Yu/m/100,-Xu/m/100,s=4,color='r',transform=proj,label='u wind points')
#plot v grid points
ax1.scatter(Yv/m/100,-Xv/m/100,s=4,color='b',transform=proj,label='v wind points')
#plot observation locations
ax1.scatter(y_obs/m/100,-x_obs/m/100,marker='o',s=20,color='k',transform=proj,label='obs locations')
plt.legend(framealpha=1.0)
ax1.set_extent(np.array([Y.min(),Y.max(),-X.min(),-X.max()])/m/100, crs=proj)

plt.show()

In [ ]:
#create analysis from observed heights & winds
#first calculate dmin (average minimum distance to closest analysis point)
ROI = 12.7775892
Kd = 10.8844524

#create analysis for H, U, and V
#U & V on the staggered u- and v-grids, respectively
H_a = 
U_a = 
V_a = 

#create analysis from observed heights and geostrophic winds
dx_m, dy_m = dx/m/100, dy/m/100
UGc, VGc = geostrophic_c(H_a, dx_m, dy_m, f, f, sig_u, sig_v)

In [ ]:
### Plot Analyses ###
#get u and v winds at the h-grid points
u_at_h, v_at_h = to_h_points(U_a,V_a)

#plot the analyses



In [ ]:
### RUN MODEL ###
dt = 60 #timestep (keep at or below 300)
last_hr = 24 #how many hours to run
g = 9.80665 #gravity

#run with unbalanced winds
H, U, V = shallow_water(H_a, U_a, V_a, X/m/100, Y/m/100, dt, last_hr, f=f, g=g,
                           sig_h=sig, sig_u=sig_u, sig_v=sig_v, advect=True)
#run with geostrophic winds
Hg, Ug, Vg = shallow_water(H_a, UGc, VGc, X/m/100, Y/m/100, dt, last_hr, f=f, g=g,
                           sig_h=sig, sig_u=sig_u, sig_v=sig_v, advect=True)

In [ ]:
### Plot Result ###
uh, vh = to_h_points(U[-1],V[-1])
ug_h, vg_h = to_h_points(Ug[-1],Vg[-1])


In [ ]:
#plot difference field

In [ ]:
#plot mean aboslute height tendency

In [ ]:
### In a separte text file (or below), answer the following questions ###
'''
1 - The only difference in the initial conditions is the wind field.  Discuss the differences in the forecast height/wind fields?
    What might this suggest about forecasts from models that are sensitive to unbalanced initial conditions?
    

2 - Contrast the height difference fields.  Does one seem smoother than the other?
    

3 - Describe the height tendency plot.  Hypothesize why there are differences.  What does this mean about forecast stability?
    Note, the advection scheme used for height tendency will have a smoothing effect in time (acts as a low-pass filter).
'''